# 02 · Adaptive contract-review agent (Phase 2)

For each contract the agent:

1. **Pre-pass (cheap):** retrieves candidate passages for the six clause families and asks the model once for a first verdict and a confidence score.
2. **Triggers (checked in code):** a family enters the loop when its confidence is below the threshold, its passages conflict, its quote contains a cross-reference ("Section 9", "Exhibit A", "notwithstanding"), it uses a defined term without the definition, its quote is not found verbatim, or its verdict is `missing` (the rubric requires a search record).
3. **Adaptive loop:** the model picks **one action per step** and says why: `search_contract`, `check_cross_reference`, `resolve_defined_term`, `lookup_rubric`, `ask_human` (non-blocking) or `finalize`.
4. **Bounds:** at most 4 steps per family, a global tool-call cap, a cost and token budget per contract, and a no-progress detector (same action and arguments twice → forced finalize and a question for a human).
5. **Trace:** every step records `trigger_reason, action, rationale, evidence_before/after, confidence_before/after, cost, elapsed`. The trace viewer at the end shows it.

Tune on the **dev contracts only** (C01, C02, C11, C12). `run_agent` refuses test contracts unless `allow_test=True` (final scoring run only).

## 0. Setup

Add secrets in Colab (🔑 on the left) and turn on notebook access:
- `OPENAI_API_KEY`: your OpenAI key
- `GH_TOKEN`: a GitHub token that can read the private repo (used only to get `contracts/` and `rubric_v1.md`)

Outside Colab, set `DATA_DIR` to your local copy of the repo.

In [ ]:
!pip -q install openai rank_bm25

import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not Path("contract-review-agent").exists():
        token = userdata.get("GH_TOKEN")
        !git clone -q https://{token}@github.com/JJJJeny/contract-review-agent.git
    DATA_DIR = Path("contract-review-agent")
else:
    DATA_DIR = Path(".")          # run from the repo folder
print(sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Settings: families, prices and loop bounds

All bounds live here so the report can cite them. Check `PRICES` against OpenAI's current pricing.

In [ ]:
from dataclasses import dataclass, field

FAMILIES = ["payment", "termination", "liability_indemnity",
            "confidentiality", "renewal", "dispute_governing_law"]

# USD per 1M tokens (input, output). Check current OpenAI pricing before reporting costs.
PRICES = {
    "gpt-5-nano": (0.05, 0.40),
    "gpt-4.1-nano": (0.10, 0.40),
    "fake": (0.0, 0.0),
}


@dataclass
class Bounds:
    confidence_threshold: float = 0.75   # pre-pass confidence below this triggers the loop
    max_steps_per_clause: int = 4        # professor: "bound the loop"
    max_tool_calls_per_contract: int = 30  # global cap across all six families
    max_cost_usd_per_contract: float = 0.05
    max_tokens_per_contract: int = 250_000
    prepass_top_k: int = 4               # chunks retrieved per family for the pre-pass
    search_top_k: int = 3                # chunks returned by search_contract
    max_section_words: int = 350         # text returned by check_cross_reference


@dataclass
class RunConfig:
    model: str = "gpt-5-nano"
    bounds: Bounds = field(default_factory=Bounds)
    use_dense: bool = True               # needs an embeddings API; falls back to sparse
    embed_model: str = "text-embedding-3-small"

## 2. Contract tools: sections, cross-references, defined terms

- `section_text("9.2")` / `section_text("Exhibit A")` back `check_cross_reference`. If an exhibit is only referred to and never filed, it returns `EXHIBIT NOT INCLUDED`.
- `define(term)` backs `resolve_defined_term`.
- `contains(quote)` is the quote check used at `finalize`.
- `cross_refs` and `undefined_terms` feed the triggers.

In [ ]:
import re
from dataclasses import dataclass, field
from pathlib import Path

# A numbered heading such as "9." "9.2" "12.1.3" at the start of a line or sentence,
# followed by a capital letter or "(" (so "30 days" or "Section 9.2" are not headings).
HEADING = re.compile(r'(?:(?<=\n)|(?<=^)|(?<=\. )|(?<=: ))[ \t]*(\d{1,2}(?:\.\d{1,2}){0,3})\.?[ \t]+(?=[A-Z(“"])')
ATTACHMENT = re.compile(r'\b(EXHIBIT|SCHEDULE|APPENDIX|ANNEX|Exhibit|Schedule|Appendix|Annex)\s+["“]?([A-Z0-9]{1,2})\b["”]?')
CROSS_REF = re.compile(
    r'(?:\b(?:Section|Sections|Paragraph|Clause|Article)\s+\d{1,2}(?:\.\d{1,2})*(?:\s*\([a-z0-9]{1,4}\))*'
    r'|§\s*\d{1,2}(?:\.\d{1,2})*'
    r'|\b(?:Exhibit|Schedule|Appendix|Annex|EXHIBIT|SCHEDULE)\s+["“]?[A-Z0-9]{1,2}\b["”]?'
    r'|\bnotwithstanding\b|\bexcept as (?:set forth|provided|otherwise)|\bsubject to (?:the|Section|Article|Paragraph))',
    re.I)
DEFINED = re.compile(r'[("“]\s*(?:the\s+)?["“]([A-Z][A-Za-z0-9\-]*(?:\s+[A-Z][A-Za-z0-9\-]*){0,4})["”]')
MEANS = re.compile(r'\b([A-Z][A-Za-z\-]*(?:\s+[A-Z][A-Za-z\-]*){0,4})["”]?\s+(?:shall\s+)?(?:means?|has the meaning)\b')
ROLE_TERMS = {"Customer", "Provider", "Recipient", "Service", "Services", "Company", "Contractor", "Client", "Vendor"}
TRIVIAL_TERMS = {"Agreement", "Party", "Parties", "Effective Date", "Section", "Exhibit", "Schedule"}


def norm(s) -> str:
    s = s or ""
    return re.sub(r"\s+", " ", s.replace("“", '"').replace("”", '"').replace("’", "'")).strip()


@dataclass
class Chunk:
    idx: int
    start: int
    end: int
    section: str
    text: str


@dataclass
class Contract:
    cid: str
    text: str
    meta: dict = field(default_factory=dict)
    headings: list = field(default_factory=list)   # [(pos, "9.2")]
    chunks: list = field(default_factory=list)
    defined_terms: dict = field(default_factory=dict)  # term -> definition sentence
    preamble_terms: set = field(default_factory=set)

    @classmethod
    def load(cls, path, meta=None, chunk_words=110, overlap=30):
        text = Path(path).read_text(encoding="utf-8", errors="replace")
        c = cls(cid=Path(path).stem, text=text, meta=meta or {})
        c.headings = [(m.start(1), m.group(1)) for m in HEADING.finditer(text)]
        c._chunk(chunk_words, overlap)
        c._index_terms()
        return c

    def section_at(self, pos: int) -> str:
        sec = ""
        for p, s in self.headings:
            if p > pos:
                break
            sec = s
        return f"§{sec}" if sec else "untitled"

    def _chunk(self, size, overlap):
        words = [(m.start(), m.end()) for m in re.finditer(r"\S+", self.text)]
        step = size - overlap
        for i in range(0, max(1, len(words)), step):
            span = words[i:i + size]
            if not span:
                break
            s, e = span[0][0], span[-1][1]
            self.chunks.append(Chunk(len(self.chunks), s, e, self.section_at(s), norm(self.text[s:e])))
            if i + size >= len(words):
                break

    def _sentence_around(self, pos: int, max_chars=600) -> str:
        left = max(self.text.rfind(". ", 0, pos), self.text.rfind("\n\n", 0, pos))
        right = self.text.find(". ", pos)
        right = len(self.text) if right == -1 else right + 1
        return norm(self.text[left + 1:right])[:max_chars]

    def _index_terms(self):
        for rx in (DEFINED, MEANS):
            for m in rx.finditer(self.text):
                term = m.group(1).strip()
                if term in TRIVIAL_TERMS or len(term) < 3 or term in self.defined_terms:
                    continue
                self.defined_terms[term] = self._sentence_around(m.start())
                if m.start() < 1500:          # parties and roles are introduced in the preamble
                    self.preamble_terms.add(term)

    def contains(self, quote: str) -> bool:
        """verify_quote: does this exact text (whitespace-normalised) appear in the contract?"""
        q = norm(quote or "").strip(' ."')
        return bool(q) and q.lower() in norm(self.text).lower()

    def section_text(self, section_id: str, max_words=350) -> dict:
        sid = section_id.strip()
        att = ATTACHMENT.search(sid)
        if att or re.match(r'(?i)(exhibit|schedule|appendix|annex)', sid):
            return self._attachment_text(sid, max_words)
        num = re.search(r'\d{1,2}(?:\.\d{1,2})*', sid)
        if not num:
            return {"found": False, "note": f"could not parse section id {section_id!r}"}
        num = num.group(0)
        hits = [i for i, (p, s) in enumerate(self.headings) if s == num]
        if not hits:
            return {"found": False, "section": num,
                    "note": "no heading with this number; the reference may be wrong or the section may be in a missing exhibit"}
        i = hits[0]
        start = self.headings[i][0]
        depth = num.count(".")
        end = len(self.text)
        for p, s in self.headings[i + 1:]:
            if s.count(".") <= depth:
                end = p
                break
        words = norm(self.text[start:end]).split()
        return {"found": True, "section": f"§{num}", "text": " ".join(words[:max_words]),
                "truncated": len(words) > max_words}

    def _attachment_text(self, sid, max_words):
        m = re.search(r'(?i)(exhibit|schedule|appendix|annex)\s+["“]?([A-Z0-9]{1,2})', sid)
        if not m:
            return {"found": False, "note": f"could not parse attachment id {sid!r}"}
        kind, label = m.group(1).lower(), m.group(2).upper()
        rx = re.compile(rf'(?i)\b{kind}\s+["“]?{re.escape(label)}\b["”]?')
        mentions, body_starts = [], []
        for h in rx.finditer(self.text):
            before = self.text[max(0, h.start() - 40):h.start()]
            ctx = norm(self.text[max(0, h.start() - 80):h.end() + 80])
            mentions.append(ctx)
            # Treat it as the attachment itself when written as a heading (upper case, or on its own line).
            after = self.text[h.end():h.end() + 3]
            is_heading = self.text[h.start():h.end()].isupper() or before.rstrip(" \t").endswith("\n")
            # Attachments are filed after the body, so ignore table-of-contents lines ("Schedule A: Fees").
            past_body = not self.headings or h.start() > self.headings[-1][0]
            if (is_heading and past_body and not after.lstrip().startswith(":")
                    and not re.search(r'(?i)(in|to|on|of|per|under|the|see)\s*$', before)):
                body_starts.append(h.start())
        if body_starts:
            start = body_starts[-1]
            words = norm(self.text[start:]).split()
            return {"found": True, "section": f"{kind.title()} {label}", "text": " ".join(words[:max_words]),
                    "truncated": len(words) > max_words}
        return {"found": False, "section": f"{kind.title()} {label}",
                "note": "EXHIBIT NOT INCLUDED: only references to it appear in the filed text",
                "mentions": mentions[:3]}

    def define(self, term: str) -> dict:
        t = term.strip(' "“”')
        for k, v in self.defined_terms.items():
            if k.lower() == t.lower():
                return {"found": True, "term": k, "definition": v}
        # fallback: first sentence where the term appears in quotes or followed by "means"
        m = re.search(rf'["“]{re.escape(t)}["”]|{re.escape(t)}\s+(?:shall\s+)?means', self.text, re.I)
        if m:
            return {"found": True, "term": t, "definition": self._sentence_around(m.start())}
        return {"found": False, "term": t, "note": "no definition found; the term may be defined in a missing exhibit"}


def cross_refs(text: str) -> list:
    return sorted({norm(m.group(0)) for m in CROSS_REF.finditer(text)})


def undefined_terms(text: str, contract: Contract, already_resolved=()) -> list:
    """Defined terms used in a span whose definition is not yet part of the evidence."""
    out = []
    for term in contract.defined_terms:
        if term in already_resolved:
            continue
        if term in contract.preamble_terms or term in ROLE_TERMS:
            continue
        if re.search(rf'\b{re.escape(term)}\b', text):
            out.append(term)
    return out

## 3. Retrieval for `search_contract`: sparse, dense or hybrid

Sparse = BM25 keywords. Dense = OpenAI embeddings. Hybrid = reciprocal rank fusion of the two. Without embeddings it falls back to sparse and reports `mode_used`.

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

TOKEN = re.compile(r"[a-z0-9]+")


def tok(s):
    return TOKEN.findall(s.lower())


class Retriever:
    def __init__(self, contract, embedder=None):
        self.c = contract
        self.bm25 = BM25Okapi([tok(ch.text) for ch in contract.chunks])
        self.embedder = embedder          # callable(list[str]) -> np.ndarray, or None
        self._vecs = None

    def _dense_scores(self, query):
        if self._vecs is None:
            self._vecs = self.embedder([ch.text for ch in self.c.chunks])
        q = self.embedder([query])[0]
        return self._vecs @ q / (np.linalg.norm(self._vecs, axis=1) * np.linalg.norm(q) + 1e-9)

    def search(self, query, mode="hybrid", k=3, exclude=()):
        """Return the top-k chunks as dicts. mode: sparse | dense | hybrid."""
        if mode in ("dense", "hybrid") and self.embedder is None:
            mode = "sparse"               # no embeddings available: degrade, and say so in the result
        sparse = self.bm25.get_scores(tok(query))
        if mode == "sparse":
            score = sparse
        else:
            dense = self._dense_scores(query)
            if mode == "dense":
                score = dense
            else:
                rank = lambda a: np.argsort(np.argsort(-a))
                score = 1 / (60 + rank(sparse)) + 1 / (60 + rank(dense))
        order = [i for i in np.argsort(-score) if i not in exclude][:k]
        return mode, [{"chunk": int(i), "section": self.c.chunks[i].section,
                       "text": self.c.chunks[i].text} for i in order]

## 4. Rubric lookup for `lookup_rubric`

Splits `rubric_v1.md` into rules such as `payment`, `payment.late_charges`, `0.4`.

In [ ]:
import re
from pathlib import Path


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


class Rubric:
    def __init__(self, path):
        self.text = Path(path).read_text(encoding="utf-8")
        self.rules = {}           # rule_id -> text
        self.family_keywords = {}
        self._parse()

    def _parse(self):
        parts = re.split(r"^(#{2,3} .+)$", self.text, flags=re.M)
        for head, body in zip(parts[1::2], parts[2::2]):
            m = re.match(r"#{2,3} (\d+(?:\.\d+)?)\.? (.+)", head)
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            self.rules[num] = f"{head}\n{body.strip()}"
            fam = title.split()[0]
            if re.fullmatch(r"[a-z_]+", fam):              # "## 1. payment" etc.
                self.rules[fam] = self.rules[num]
                kw = re.search(r"\*\*Keywords:\*\* (.+)", body)
                if kw:
                    self.family_keywords[fam] = kw.group(1).rstrip(".")
                for row in re.findall(r"^\| ([^|]+?) \| (.+)$", body, flags=re.M):
                    name = row[0].strip()
                    if name.startswith("Sub-criterion") or set(name) <= set("-"):
                        continue
                    self.rules[f"{fam}.{slug(name)}"] = f"| {name} | {row[1]}"
                for label in ("Missing", "Severity", "Absence rule", "Customer is not the payer"):
                    p = re.search(rf"^\*\*{label}[^*]*\*\*.*$", body, flags=re.M)
                    if p:
                        self.rules[f"{fam}.{slug(label)}"] = p.group(0)

    def family_rules(self, family):
        return self.rules.get(family, "")

    def general_rules(self):
        return "\n\n".join(self.rules[k] for k in ("0.2", "0.3", "0.4") if k in self.rules)

    def lookup(self, rule_id):
        rid = rule_id.strip().lstrip("§").lower()
        if rid in self.rules:
            return {"found": True, "rule_id": rid, "text": self.rules[rid]}
        near = [k for k in self.rules if rid.split(".")[0] in k][:12]
        return {"found": False, "rule_id": rid, "available": near or sorted(self.rules)[:20]}

## 5. Model clients

`OpenAIChat` calls the API in JSON mode and counts tokens for the budget. `FakeLLM` is a scripted stand-in so the controller can be tested for free.

In [ ]:
import json
import re
import time

import numpy as np



class Usage:
    def __init__(self, model):
        self.model = model
        self.input_tokens = self.output_tokens = self.calls = 0

    def add(self, i, o):
        self.input_tokens += i
        self.output_tokens += o
        self.calls += 1
        return self.cost_of(i, o)

    def cost_of(self, i, o):
        pin, pout = PRICES.get(self.model, (0.0, 0.0))
        return (i * pin + o * pout) / 1e6

    @property
    def tokens(self):
        return self.input_tokens + self.output_tokens

    @property
    def cost(self):
        return self.cost_of(self.input_tokens, self.output_tokens)


def parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group(0)) if m else {}


class OpenAIChat:
    """Chat Completions in JSON mode. Works for gpt-5-nano / gpt-4.1-nano and other OpenAI chat models."""

    def __init__(self, model="gpt-5-nano", api_key=None, retries=3):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.retries = retries

    def complete(self, system, user, purpose=""):
        """Return (parsed_json, input_tokens, output_tokens)."""
        for attempt in range(self.retries):
            try:
                r = self.client.chat.completions.create(
                    model=self.model,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": user}],
                )
                u = r.usage
                return parse_json(r.choices[0].message.content or "{}"), u.prompt_tokens, u.completion_tokens
            except Exception:
                if attempt == self.retries - 1:
                    raise
                time.sleep(2 ** attempt)

    def embedder(self, model="text-embedding-3-small"):
        def embed(texts):
            out = []
            for i in range(0, len(texts), 100):
                r = self.client.embeddings.create(model=model, input=texts[i:i + 100])
                out += [d.embedding for d in r.data]
            return np.array(out, dtype=np.float32)
        return embed


class FakeLLM:
    """Scripted stand-in that exercises every branch of the controller (no API key needed).

    Pre-pass: quotes the first sentence of the top retrieved chunk and reports low confidence for
    some families. Loop: chases the first cross-reference, then resolves a defined term, then
    repeats an action once (to trip the no-progress detector), otherwise finalizes.
    """
    model = "fake"

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def complete(self, system, user, purpose=""):
        ctx = json.loads(user[user.index("{"):]) if "{" in user else {}
        if purpose == "prepass":
            out = {}
            for fam, chunks in ctx.get("candidates", {}).items():
                txt = chunks[0]["text"] if chunks else ""
                quote = re.split(r"(?<=[.;])\s", txt)[0][:300]
                out[fam] = {"verdict": "risky" if self.rng.random() < 0.5 else "acceptable",
                            "severity": "medium", "evidence_text": quote,
                            "section_ref": chunks[0]["section"] if chunks else "",
                            "confidence": round(float(self.rng.uniform(0.4, 0.95)), 2),
                            "spans_conflict": bool(self.rng.random() < 0.15),
                            "notes": "fake pre-pass"}
            return {"families": out}, 900, 400
        if purpose == "step":
            hist = ctx.get("history", [])
            pending = ctx.get("open_triggers", {})
            n = len(hist)
            draft = ctx.get("draft", {})
            conf = min(0.95, float(draft.get("confidence", 0.5)) + 0.15)
            if n == 0 and pending.get("cross_refs"):
                act, args = "check_cross_reference", {"section_id": pending["cross_refs"][0]}
            elif n <= 1 and pending.get("undefined_terms"):
                act, args = "resolve_defined_term", {"term": pending["undefined_terms"][0]}
            elif n == 2 and hist[-1]["action"] != "finalize":
                act, args = hist[-1]["action"], hist[-1]["args"]          # deliberate repeat
            elif n == 0:
                act, args = "search_contract", {"query": ctx["family"].replace("_", " "), "mode": "hybrid"}
            else:
                ev = ctx.get("evidence", [{}])
                act, args = "finalize", {"verdict": draft.get("verdict", "unclear"),
                                         "severity": draft.get("severity", "low"),
                                         "evidence_text": ev[0].get("text", "") if ev else "",
                                         "section_ref": ev[0].get("section", "") if ev else "",
                                         "notes": "fake finalize"}
            return {"rationale": f"fake: step {n + 1} chose {act}", "action": act, "args": args,
                    "confidence": round(conf, 2),
                    "draft_verdict": draft.get("verdict", "unclear"),
                    "draft_severity": draft.get("severity", "low")}, 1200, 150
        if purpose == "plan":
            return {"items": [{"id": f["id"], "ask": "fake ask", "fallback": "fake fallback"}
                              for f in ctx.get("findings", [])]}, 600, 200
        return {}, 0, 0

    def embedder(self, model=None):
        def embed(texts):                 # hashed bag of words, enough to test the dense path
            v = np.zeros((len(texts), 256), dtype=np.float32)
            for r, t in enumerate(texts):
                for w in re.findall(r"[a-z]+", t.lower()):
                    v[r, hash(w) % 256] += 1
            return v
        return embed

## 6. Prompts

In [ ]:
PERSPECTIVE = (
    "You review vendor service agreements for a SMALL-BUSINESS CUSTOMER that buys maintenance or "
    "services. It has no lawyer and little bargaining power. Judge every clause from the customer's "
    "side, even when the provider filed the contract. Apply the rubric exactly; do not invent thresholds."
)

PREPASS_SYSTEM = PERSPECTIVE + """

This is a quick first pass. For EACH of the six families you get a few candidate passages retrieved
from the contract. Give a first verdict, but be honest about confidence: lower it when the passages
do not show the whole clause, refer to other sections or exhibits, use defined terms you cannot see,
or point in different directions.

Rules that apply to every family:
{general}

Family rules:
{families}

Return JSON only:
{{"families": {{"<family>": {{
  "verdict": "acceptable|risky|missing|unclear",
  "severity": "high|medium|low|n/a",
  "evidence_text": "exact sentence copied from a passage (empty if missing)",
  "section_ref": "section as printed, e.g. §9.2",
  "evidence_2": "second exact quote the verdict depends on, or empty",
  "section_ref_2": "",
  "confidence": 0.0-1.0,
  "spans_conflict": true|false,
  "notes": "one or two sentences: which sub-criterion decided it"
}}}}}}
Include all six families."""

STEP_SYSTEM = PERSPECTIVE + """

You are now checking ONE clause family because the first pass was not reliable enough. The
"open_triggers" field says why. Each turn choose exactly ONE action:

- search_contract {{"query": str, "mode": "sparse"|"dense"|"hybrid"}}: re-retrieve with different words or mode
- check_cross_reference {{"section_id": str}}: read a referenced section or exhibit, e.g. "9.2", "Exhibit A"
- resolve_defined_term {{"term": str}}: pull the definition of a capitalized term
- lookup_rubric {{"rule_id": str}}: re-read a rule, e.g. "{family}.payment_window", "0.4"
- ask_human {{"question": str}}: queue a question for the human reviewer; does not stop you
- finalize {{"verdict", "severity", "evidence_text", "section_ref", "evidence_2", "section_ref_2", "notes"}}: stop

Rules:
- Prefer the action that resolves an open trigger. Do not repeat an action with the same arguments.
- evidence_text and evidence_2 must be copied EXACTLY from the evidence you have seen; they are checked.
- A "missing" verdict needs a search record in notes (keywords tried, sections checked).
- An exhibit or schedule that is not in the filed text makes the dependent term "unclear" (usually low).
- When steps_left is 1 you MUST finalize.

Rubric for this family:
{rules}

Return JSON only:
{{"rationale": "why this action now, naming the trigger it addresses",
  "action": "<one action name>", "args": {{...}},
  "confidence": 0.0-1.0, "draft_verdict": "...", "draft_severity": "..."}}"""

PLAN_SYSTEM = PERSPECTIVE + """

Write the negotiation ask for each finding. For each item give one plain sentence the customer can
say to the provider ("ask") and a fallback if the provider says no. Return JSON only:
{"items": [{"id": int, "ask": str, "fallback": str}]}"""

## 7. The agent: pre-pass → triggers → bounded loop → revision plan

`ContractRun.loop` is the controller. The model only proposes the next action; the code decides whether the loop starts, enforces every bound, rejects unverified quotes, and logs each step.

In [ ]:
import json
import time


ACTIONS = {"search_contract", "check_cross_reference", "resolve_defined_term",
           "lookup_rubric", "ask_human", "finalize"}
VERDICTS = {"acceptable", "risky", "missing", "unclear"}
SEVERITIES = {"high", "medium", "low", "n/a"}
SEV_ORDER = {"high": 0, "medium": 1, "low": 2, "n/a": 3}
VERDICT_ORDER = {"risky": 0, "missing": 1, "unclear": 2, "acceptable": 3}
FAMILY_TIEBREAK = ["liability_indemnity", "termination", "payment", "renewal",
                   "confidentiality", "dispute_governing_law"]


def short(ev, n=90):
    return [f'{e.get("section", "")}: {norm(e.get("text", ""))[:n]}' for e in ev]


class BudgetExceeded(Exception):
    pass


class ContractRun:
    def __init__(self, contract, rubric, llm, cfg=None, embedder=None):
        self.c, self.rubric, self.llm = contract, rubric, llm
        self.cfg = cfg or RunConfig()
        self.b = self.cfg.bounds
        self.retriever = Retriever(contract, embedder)
        self.usage = Usage(getattr(llm, "model", self.cfg.model))
        self.tool_calls = 0
        self.trace = []           # one dict per step, all families
        self.human_queue = []
        self.results = {}
        self.plan = []

    # ---------- budget ----------
    def _call(self, system, user, purpose):
        if self.over_budget():
            raise BudgetExceeded(self.over_budget())
        out, i, o = self.llm.complete(system, user, purpose)
        return out, self.usage.add(i, o), i + o

    def over_budget(self):
        if self.tool_calls >= self.b.max_tool_calls_per_contract:
            return "global tool-call cap"
        if self.usage.cost >= self.b.max_cost_usd_per_contract:
            return "cost budget"
        if self.usage.tokens >= self.b.max_tokens_per_contract:
            return "token budget"
        return None

    def _log(self, **kw):
        kw.setdefault("contract_id", self.c.cid)
        self.trace.append(kw)

    # ---------- pre-pass ----------
    def prepass(self):
        cands = {}
        for fam in FAMILIES:
            q = self.rubric.family_keywords.get(fam, fam.replace("_", " "))
            _, hits = self.retriever.search(q, "hybrid", self.b.prepass_top_k)
            cands[fam] = hits
        system = PREPASS_SYSTEM.format(general=self.rubric.general_rules(),
                                       families="\n\n".join(self.rubric.family_rules(f) for f in FAMILIES))
        user = "Contract " + self.c.cid + " (" + self.c.meta.get("short_name", "") + "):\n" + \
            json.dumps({"contract_id": self.c.cid, "candidates": cands}, ensure_ascii=False)
        t0 = time.time()
        out, cost, tokens = self._call(system, user, "prepass")
        fams = out.get("families", {})
        for fam in FAMILIES:
            d = fams.get(fam, {}) or {}
            d.setdefault("verdict", "unclear")
            d.setdefault("confidence", 0.0)
            self.results[fam] = {"prepass": d, "candidates": cands[fam]}
        self._log(family="*", step=0, phase="prepass", trigger_reason="", action="prepass",
                  args={"families": FAMILIES}, rationale="cheap extraction pass over retrieved candidates",
                  evidence_before=[], evidence_after=[], confidence_before=None,
                  confidence_after={f: self.results[f]["prepass"].get("confidence") for f in FAMILIES},
                  cost=round(cost, 6), tokens=tokens, elapsed=round(time.time() - t0, 2))

    # ---------- triggers (computed in code, not by the model) ----------
    def triggers(self, fam, draft, evidence, chased, resolved):
        # Only the decisive quotes and sections reached by following a reference are scanned;
        # broad search hits would mention dozens of terms and keep the loop firing.
        quotes = " ".join(e.get("text", "") for e in evidence if e.get("source") in ("prepass", "cross_ref"))
        t = {}
        if float(draft.get("confidence") or 0) < self.b.confidence_threshold:
            t["low_confidence"] = draft.get("confidence")
        if draft.get("spans_conflict"):
            t["spans_conflict"] = True
        refs = [r for r in cross_refs(quotes) if r.lower() not in chased]
        if refs:
            t["cross_refs"] = refs[:5]
        terms = undefined_terms(quotes, self.c, resolved)
        if terms:
            t["undefined_terms"] = terms[:5]
        for key in ("evidence_text", "evidence_2"):
            q = draft.get(key)
            if q and not self.c.contains(q):
                t["quote_not_found"] = key
        if draft.get("verdict") == "missing":
            t["missing_needs_search"] = True
        return t

    # ---------- tools ----------
    def run_tool(self, action, args, state):
        if action == "search_contract":
            mode, hits = self.retriever.search(str(args.get("query", "")), args.get("mode", "hybrid"),
                                               self.b.search_top_k)
            state["evidence"] += [{**h, "source": "search"} for h in hits]
            state["searches"].append(args.get("query", ""))
            return {"mode_used": mode, "hits": hits}
        if action == "check_cross_reference":
            sid = str(args.get("section_id", ""))
            state["chased"].add(sid.lower())
            for r in list(state["open"].get("cross_refs", [])):
                if sid.lower().replace("section", "").strip(" §") in r.lower():
                    state["chased"].add(r.lower())
            res = self.c.section_text(sid, self.b.max_section_words)
            if res.get("found"):
                state["evidence"].append({"section": res["section"], "text": res["text"], "source": "cross_ref"})
            return res
        if action == "resolve_defined_term":
            term = str(args.get("term", ""))
            state["resolved"].add(term)
            res = self.c.define(term)
            if res.get("found"):
                state["evidence"].append({"section": f'definition of "{res["term"]}"',
                                          "text": res["definition"], "source": "definition"})
            return res
        if action == "lookup_rubric":
            return self.rubric.lookup(str(args.get("rule_id", "")))
        if action == "ask_human":
            q = {"contract_id": self.c.cid, "family": state["family"], "question": args.get("question", ""),
                 "draft": dict(state["draft"])}
            self.human_queue.append(q)
            return {"queued": True, "position": len(self.human_queue)}
        raise ValueError(action)

    def _finalize(self, fam, args, path, state):
        v = args.get("verdict") if args.get("verdict") in VERDICTS else "unclear"
        s = args.get("severity") if args.get("severity") in SEVERITIES else "low"
        if v == "acceptable":
            s = "n/a"
        row = {"verdict": v, "severity": s,
               "evidence_text": args.get("evidence_text", "") if v != "missing" else (args.get("evidence_text") or "NONE FOUND"),
               "section_ref": args.get("section_ref", ""), "evidence_2": args.get("evidence_2", ""),
               "section_ref_2": args.get("section_ref_2", ""), "notes": args.get("notes", ""),
               "confidence": state["draft"].get("confidence"), "path": path,
               "steps": state["steps"], "searches": state["searches"]}
        for key in ("evidence_text", "evidence_2"):
            q = row[key]
            row[key + "_verified"] = (q in ("", "NONE FOUND")) or self.c.contains(q)
        self.results[fam]["final"] = row
        return row

    def _step_payload(self, fam, state, steps_left):
        return {"family": fam, "contract_id": self.c.cid, "steps_left": steps_left,
                "open_triggers": state["open"], "draft": state["draft"],
                "evidence": [{"section": e["section"], "text": e["text"][:1200], "source": e.get("source", "")}
                             for e in state["evidence"][-8:]],
                "history": [{"action": h["action"], "args": h["args"], "result": h["result"]}
                            for h in state["history"]],
                "rubric_rule_ids": sorted(k for k in self.rubric.rules if k.startswith(fam + "."))}

    def loop(self, fam):
        pre = self.results[fam]["prepass"]
        evidence = []
        for key, ref in (("evidence_text", "section_ref"), ("evidence_2", "section_ref_2")):
            if pre.get(key):
                evidence.append({"section": pre.get(ref, ""), "text": pre[key], "source": "prepass"})
        state = {"family": fam, "evidence": evidence, "history": [], "searches": [], "steps": 0,
                 "chased": set(), "resolved": set(),
                 "draft": {"verdict": pre.get("verdict"), "severity": pre.get("severity"),
                           "confidence": pre.get("confidence"), "spans_conflict": pre.get("spans_conflict"),
                           "evidence_text": pre.get("evidence_text"), "section_ref": pre.get("section_ref"),
                           "evidence_2": pre.get("evidence_2"), "section_ref_2": pre.get("section_ref_2")}}
        state["open"] = self.triggers(fam, state["draft"], evidence, set(), set())
        if not state["open"]:
            self._log(family=fam, step=0, phase="no_loop", trigger_reason="none", action="finalize",
                      args={}, rationale="no trigger fired; pre-pass verdict kept",
                      evidence_before=short(evidence), evidence_after=short(evidence),
                      confidence_before=pre.get("confidence"), confidence_after=pre.get("confidence"),
                      cost=0.0, tokens=0, elapsed=0.0)
            return self._finalize(fam, pre, "prepass", state)

        system = STEP_SYSTEM.format(family=fam, rules=self.rubric.family_rules(fam))
        seen = set()
        for step in range(1, self.b.max_steps_per_clause + 1):
            state["steps"] = step
            steps_left = self.b.max_steps_per_clause - step + 1
            ev_before, conf_before = short(state["evidence"]), state["draft"].get("confidence")
            trig = ", ".join(sorted(state["open"])) or "none"
            t0 = time.time()
            try:
                out, cost, tokens = self._call(system, "State:\n" + json.dumps(
                    self._step_payload(fam, state, steps_left), ensure_ascii=False, default=list), "step")
            except BudgetExceeded as e:
                self._log(family=fam, step=step, phase="forced", trigger_reason=f"budget: {e}",
                          action="finalize", args={}, rationale=f"{e} reached; keeping current draft",
                          evidence_before=ev_before, evidence_after=ev_before, confidence_before=conf_before,
                          confidence_after=conf_before, cost=0.0, tokens=0, elapsed=0.0)
                return self._finalize(fam, {**state["draft"], "notes": f"forced finalize ({e})"},
                                      f"forced:{e}", state)
            action, args = out.get("action"), out.get("args") or {}
            state["draft"]["confidence"] = out.get("confidence", conf_before)
            if out.get("draft_verdict") in VERDICTS:
                state["draft"]["verdict"] = out["draft_verdict"]
            if out.get("draft_severity") in SEVERITIES:
                state["draft"]["severity"] = out["draft_severity"]
            key = (action, json.dumps(args, sort_keys=True))
            forced = None
            if action not in ACTIONS:
                forced = f"invalid action {action!r}"
            elif key in seen and action != "finalize":
                forced = "no progress: same action and args repeated"
            elif steps_left == 1 and action != "finalize":
                forced = "step limit reached"
            if forced:
                self.human_queue.append({"contract_id": self.c.cid, "family": fam,
                                         "question": f"Agent stopped ({forced}). Please check this verdict.",
                                         "draft": dict(state["draft"])})
                self._log(family=fam, step=step, phase="forced", trigger_reason=trig, action=action,
                          args=args, rationale=out.get("rationale", ""), forced=forced,
                          evidence_before=ev_before, evidence_after=ev_before, confidence_before=conf_before,
                          confidence_after=state["draft"]["confidence"], cost=round(cost, 6), tokens=tokens,
                          elapsed=round(time.time() - t0, 2))
                best = {**state["draft"], **{k: v for k, v in args.items() if action == "finalize"}}
                if not best.get("evidence_text") and state["evidence"]:
                    best["evidence_text"] = state["evidence"][0]["text"]
                    best["section_ref"] = state["evidence"][0]["section"]
                best["notes"] = (best.get("notes") or "") + f" [forced finalize: {forced}; queued for human]"
                return self._finalize(fam, best, f"forced:{forced}", state)
            seen.add(key)
            self.tool_calls += 1

            if action == "finalize":
                bad = [k for k in ("evidence_text", "evidence_2")
                       if args.get(k) and args.get(k) != "NONE FOUND" and not self.c.contains(args[k])]
                if bad and steps_left > 1:
                    result = {"rejected": True, "reason": f"{', '.join(bad)} not found verbatim in the contract"}
                    state["history"].append({"action": action, "args": args, "result": result})
                    state["open"]["quote_not_found"] = bad[0]
                    self._log(family=fam, step=step, phase="loop", trigger_reason=trig, action=action, args=args,
                              rationale=out.get("rationale", ""), result=result, evidence_before=ev_before,
                              evidence_after=ev_before, confidence_before=conf_before,
                              confidence_after=state["draft"]["confidence"], cost=round(cost, 6), tokens=tokens,
                              elapsed=round(time.time() - t0, 2))
                    continue
                self._log(family=fam, step=step, phase="loop", trigger_reason=trig, action=action, args=args,
                          rationale=out.get("rationale", ""), result={"accepted": True},
                          evidence_before=ev_before, evidence_after=ev_before, confidence_before=conf_before,
                          confidence_after=state["draft"]["confidence"], cost=round(cost, 6), tokens=tokens,
                          elapsed=round(time.time() - t0, 2))
                return self._finalize(fam, args, "loop", state)

            result = self.run_tool(action, args, state)
            state["history"].append({"action": action, "args": args,
                                     "result": json.loads(json.dumps(result, default=str))})
            state["open"] = self.triggers(fam, state["draft"], state["evidence"], state["chased"], state["resolved"])
            self._log(family=fam, step=step, phase="loop", trigger_reason=trig, action=action, args=args,
                      rationale=out.get("rationale", ""), result=result,
                      evidence_before=ev_before, evidence_after=short(state["evidence"]),
                      confidence_before=conf_before, confidence_after=state["draft"]["confidence"],
                      cost=round(cost, 6), tokens=tokens, elapsed=round(time.time() - t0, 2))

    # ---------- revision plan (§7) ----------
    def revision_plan(self):
        findings = []
        for fam in FAMILIES:
            r = self.results[fam]["final"]
            if r["verdict"] != "acceptable":
                findings.append({"family": fam, **{k: r[k] for k in ("verdict", "severity", "evidence_text",
                                                                    "section_ref", "notes")}})
        findings.sort(key=lambda f: (SEV_ORDER.get(f["severity"], 3), VERDICT_ORDER[f["verdict"]],
                                     FAMILY_TIEBREAK.index(f["family"])))
        for i, f in enumerate(findings, 1):
            f["id"] = i
        if findings and not self.over_budget():
            try:
                out, _, _ = self._call(PLAN_SYSTEM, json.dumps({"findings": findings}, ensure_ascii=False), "plan")
                asks = {it.get("id"): it for it in out.get("items", [])}
                for f in findings:
                    f["ask"] = asks.get(f["id"], {}).get("ask", "")
                    f["fallback"] = asks.get(f["id"], {}).get("fallback", "")
            except BudgetExceeded:
                pass
        self.plan = findings
        return findings

    def run(self):
        t0 = time.time()
        self.prepass()
        for fam in FAMILIES:
            self.loop(fam)
        self.revision_plan()
        self.elapsed = round(time.time() - t0, 2)
        return self

## 8. Trace viewer (one HTML file for debugging, the report and the demo)

In [ ]:
import csv
import json
from pathlib import Path

PAGE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Agent trace viewer</title>
<style>
:root{--bg:#fafaf8;--fg:#1d1d1b;--mut:#6b6b66;--line:#e2e1dc;--card:#fff;--risky:#b3261e;--unclear:#9a6700;--missing:#6f42c1;--ok:#1a7f37;--forced:#fff4e5;--acc:#0b57d0}
@media (prefers-color-scheme:dark){:root{--bg:#161615;--fg:#ecebe6;--mut:#9d9c95;--line:#33322f;--card:#1f1f1d;--risky:#f2867d;--unclear:#e3b341;--missing:#b89cf0;--ok:#56c26f;--forced:#3a2a12;--acc:#8ab4f8}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--fg);font:14px/1.45 system-ui,-apple-system,Segoe UI,sans-serif}
main{max-width:1200px;margin:0 auto;padding:16px}h1{font-size:20px;margin:0 0 4px}h2{font-size:16px;margin:24px 0 8px}
.mut{color:var(--mut)}select{font:inherit;padding:4px 8px;background:var(--card);color:var(--fg);border:1px solid var(--line);border-radius:6px}
.stats{display:flex;flex-wrap:wrap;gap:8px;margin:12px 0}.stat{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:8px 12px}
.stat b{display:block;font-size:18px}.wrap{overflow-x:auto}table{border-collapse:collapse;width:100%;background:var(--card)}
th,td{border-bottom:1px solid var(--line);padding:6px 8px;text-align:left;vertical-align:top}th{font-size:12px;color:var(--mut);font-weight:600}
tr.fam{cursor:pointer}tr.fam:hover{background:color-mix(in srgb,var(--acc) 8%,transparent)}tr.sel{outline:2px solid var(--acc)}
.v-risky{color:var(--risky);font-weight:600}.v-unclear{color:var(--unclear);font-weight:600}.v-missing{color:var(--missing);font-weight:600}.v-acceptable{color:var(--ok);font-weight:600}
tr.forced td{background:var(--forced)}code{font-size:12px;white-space:pre-wrap;word-break:break-word}.ev{font-size:12px;color:var(--mut);max-width:360px}
.pill{display:inline-block;border:1px solid var(--line);border-radius:10px;padding:0 6px;font-size:12px;margin:1px}
</style></head><body><main>
<h1>Agent trace viewer</h1><div class="mut" id="meta"></div>
<p>Contract <select id="pick"></select></p>
<div class="stats" id="stats"></div>
<h2>Verdicts <span class="mut">(click a row to see its steps)</span></h2><div class="wrap"><table id="res"></table></div>
<h2 id="steps-h">Steps</h2><div class="wrap"><table id="steps"></table></div>
<h2>Revision plan</h2><div class="wrap"><table id="plan"></table></div>
<h2>Questions queued for a human</h2><div class="wrap"><table id="queue"></table></div>
</main><script>
const D=__DATA__;
const esc=s=>String(s??"").replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const pick=document.getElementById("pick");
D.ids.forEach(id=>pick.add(new Option(id+" "+(D.rows.find(r=>r.contract_id==id)||{}).short_name,id)));
document.getElementById("meta").textContent=`model ${D.summary.model} · bounds: ${D.summary.bounds.max_steps_per_clause} steps/clause, ${D.summary.bounds.max_tool_calls_per_contract} tool calls/contract, $${D.summary.bounds.max_cost_usd_per_contract}/contract, confidence < ${D.summary.bounds.confidence_threshold} triggers the loop`;
let fam=null;
function render(){
 const id=pick.value,s=D.summary.contracts.find(c=>c.contract_id==id)||{};
 document.getElementById("stats").innerHTML=[["families looped",s.families_looped+" / 6"],["LLM calls",s.llm_calls],["tool calls",s.tool_calls],["tokens",s.tokens],["cost (USD)",s.cost_usd],["time (s)",s.elapsed_s]].map(([k,v])=>`<div class="stat"><b>${esc(v)}</b>${k}</div>`).join("");
 const rows=D.rows.filter(r=>r.contract_id==id);
 document.getElementById("res").innerHTML="<tr><th>family</th><th>verdict</th><th>severity</th><th>confidence</th><th>path</th><th>steps</th><th>quote ok</th><th>evidence</th><th>notes</th></tr>"+rows.map(r=>`<tr class="fam ${r.family==fam?"sel":""}" data-f="${r.family}"><td>${r.family}</td><td class="v-${r.verdict}">${r.verdict}</td><td>${r.severity}</td><td>${esc(r.confidence)}</td><td>${esc(r.path)}</td><td>${r.steps}</td><td>${r.evidence_verified=="True"?"✓":"✗"}</td><td class="ev">${esc(r.section_ref)} ${esc(r.evidence_text).slice(0,220)}</td><td class="ev">${esc(r.notes)}</td></tr>`).join("");
 document.querySelectorAll("tr.fam").forEach(tr=>tr.onclick=()=>{fam=tr.dataset.f;render()});
 const st=D.trace.filter(t=>t.contract_id==id&&(fam?t.family==fam||t.family=="*":true));
 document.getElementById("steps-h").textContent="Steps"+(fam?" · "+fam:" · all families");
 document.getElementById("steps").innerHTML="<tr><th>family</th><th>step</th><th>trigger</th><th>action</th><th>rationale</th><th>confidence</th><th>new evidence</th><th>cost</th><th>s</th></tr>"+st.map(t=>{const added=(t.evidence_after||[]).filter(e=>!(t.evidence_before||[]).includes(e));const res=t.result?JSON.stringify(t.result).slice(0,300):"";return `<tr class="${t.forced||t.phase=="forced"?"forced":""}"><td>${t.family}</td><td>${t.step}</td><td>${esc(t.trigger_reason).split(", ").map(x=>x?`<span class="pill">${esc(x)}</span>`:"").join("")}</td><td><b>${esc(t.action)}</b><br><code>${esc(JSON.stringify(t.args))}</code></td><td>${esc(t.rationale)}${t.forced?`<br><b>forced:</b> ${esc(t.forced)}`:""}${res?`<br><code class="mut">${esc(res)}</code>`:""}</td><td>${typeof t.confidence_after=="object"&&t.confidence_after?Object.entries(t.confidence_after).map(([k,v])=>k+": "+v).join("<br>"):esc(t.confidence_before)+" → "+esc(t.confidence_after)}</td><td class="ev">${added.map(esc).join("<br>")}</td><td>${t.cost}</td><td>${t.elapsed}</td></tr>`}).join("");
 const p=D.plans[id]||[];
 document.getElementById("plan").innerHTML="<tr><th>#</th><th>family</th><th>finding</th><th>evidence</th><th>ask</th><th>fallback</th></tr>"+p.map(f=>`<tr><td>${f.id}</td><td>${f.family}</td><td class="v-${f.verdict}">${f.verdict} · ${f.severity}</td><td class="ev">${esc(f.section_ref)} ${esc(f.evidence_text).slice(0,200)}</td><td>${esc(f.ask)}</td><td>${esc(f.fallback)}</td></tr>`).join("");
 const q=D.queue.filter(x=>x.contract_id==id);
 document.getElementById("queue").innerHTML="<tr><th>family</th><th>question</th><th>agent draft</th></tr>"+(q.length?q.map(x=>`<tr><td>${x.family}</td><td>${esc(x.question)}</td><td>${esc(x.draft&&x.draft.verdict)} / ${esc(x.draft&&x.draft.severity)}</td></tr>`).join(""):'<tr><td colspan="3" class="mut">none</td></tr>');
}
pick.onchange=()=>{fam=null;render()};render();
</script></body></html>"""


def write_viewer(run_dir):
    run_dir = Path(run_dir)
    with open(run_dir / "results.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    data = {
        "rows": rows,
        "ids": list(dict.fromkeys(r["contract_id"] for r in rows)),
        "trace": [json.loads(l) for l in open(run_dir / "trace.jsonl")],
        "queue": json.loads((run_dir / "human_queue.json").read_text()),
        "plans": json.loads((run_dir / "revision_plans.json").read_text()),
        "summary": json.loads((run_dir / "summary.json").read_text()),
    }
    blob = json.dumps(data, ensure_ascii=False, default=str).replace("</", "<\\/")
    out = run_dir / "trace_viewer.html"
    out.write_text(PAGE.replace("__DATA__", blob), encoding="utf-8")
    return out

## 9. Runner

In [ ]:
import csv
import json
from pathlib import Path

COLUMNS = ["contract_id", "short_name", "split", "family", "verdict", "severity", "evidence_text",
           "section_ref", "evidence_2", "section_ref_2", "notes",
           "confidence", "path", "steps", "evidence_verified"]


def load_meta():
    with open(DATA_DIR / "contracts.csv", newline="") as f:
        return {r["contract_id"]: r for r in csv.DictReader(f)}


def run_agent(ids, llm, cfg=None, out_dir="runs/latest", use_dense=True, allow_test=False):
    """Run the agent on contracts `ids`; write results, traces, human queue, plans and the viewer."""
    cfg = cfg or RunConfig()
    meta = load_meta()
    test_ids = [i for i in ids if meta[i]["split"] == "test"]
    if test_ids and not allow_test:
        raise ValueError(f"{test_ids} are test contracts. Tune on dev only; pass allow_test=True for the final scoring run.")
    rubric = Rubric(DATA_DIR / "rubric_v1.md")
    embedder = None
    if use_dense and cfg.use_dense:
        try:
            embedder = llm.embedder(cfg.embed_model)
        except Exception as e:                       # no embeddings: search degrades to sparse
            print("dense retrieval off:", e)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, traces, queue, plans, summary = [], [], [], {}, []
    for cid in ids:
        c = Contract.load(DATA_DIR / "contracts" / f"{cid}.txt", meta[cid])
        r = ContractRun(c, rubric, llm, cfg, embedder).run()
        for fam in FAMILIES:
            f = r.results[fam]["final"]
            rows.append({"contract_id": cid, "short_name": meta[cid]["short_name"], "split": meta[cid]["split"],
                         "family": fam, **{k: f.get(k, "") for k in COLUMNS[4:11]},
                         "confidence": f.get("confidence"), "path": f["path"], "steps": f["steps"],
                         "evidence_verified": f["evidence_text_verified"] and f["evidence_2_verified"]})
        traces += r.trace
        queue += r.human_queue
        plans[cid] = r.plan
        summary.append({"contract_id": cid, "llm_calls": r.usage.calls, "tool_calls": r.tool_calls,
                        "tokens": r.usage.tokens, "cost_usd": round(r.usage.cost, 5), "elapsed_s": r.elapsed,
                        "families_looped": sum(1 for fam in FAMILIES if r.results[fam]["final"]["path"] != "prepass")})
        print(summary[-1])
    with open(out_dir / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, COLUMNS)
        w.writeheader()
        w.writerows(rows)
    with open(out_dir / "trace.jsonl", "w") as f:
        for t in traces:
            f.write(json.dumps(t, ensure_ascii=False, default=list) + "\n")
    (out_dir / "human_queue.json").write_text(json.dumps(queue, indent=2, ensure_ascii=False, default=list))
    (out_dir / "revision_plans.json").write_text(json.dumps(plans, indent=2, ensure_ascii=False))
    (out_dir / "summary.json").write_text(json.dumps({"model": getattr(llm, "model", ""), "contracts": summary,
                                                      "bounds": vars(cfg.bounds)}, indent=2))
    write_viewer(out_dir)
    print("wrote", out_dir)
    return out_dir

## 10. Controller checks with the fake model (free)

Checks the step limit, the no-progress detector, the global tool cap and the cross-reference tool. Run this after any change to the controller.

In [ ]:
class StuckLLM(FakeLLM):
    """Always asks for the same search, so every looped family should hit the no-progress detector."""

    def complete(self, system, user, purpose=""):
        if purpose == "step":
            return {"rationale": "stuck", "action": "search_contract",
                    "args": {"query": "fees", "mode": "sparse"}, "confidence": 0.5}, 100, 20
        return super().complete(system, user, purpose)


def check_run(cid="C12", llm=None, **b):
    llm = llm or FakeLLM()
    c = Contract.load(DATA_DIR / "contracts" / f"{cid}.txt")
    return ContractRun(c, Rubric(DATA_DIR / "rubric_v1.md"), llm,
                       RunConfig(model="fake", bounds=Bounds(**b)), llm.embedder()).run()


r = check_run()
assert set(r.results) == set(FAMILIES)
keys = {"step", "trigger_reason", "action", "rationale", "evidence_before", "evidence_after",
        "confidence_before", "confidence_after", "cost", "elapsed"}
assert all(keys <= set(t) for t in r.trace), "every step must be logged with the full record"
assert max(check_run(max_steps_per_clause=2).results[f]["final"]["steps"] for f in FAMILIES) <= 2
r = check_run("C01", llm=StuckLLM())
assert any(t.get("forced", "").startswith("no progress") for t in r.trace)
assert any("no progress" in q["question"] for q in r.human_queue)
r = check_run(max_tool_calls_per_contract=3)
assert r.tool_calls <= 3 and any(t["trigger_reason"].startswith("budget") for t in r.trace)
c = Contract.load(DATA_DIR / "contracts" / "C01.txt")
assert c.section_text("Section 17(b)")["section"] == "§17"
assert c.section_text("Schedule A")["found"] is False      # C01's fee schedule was never filed
print("all controller checks passed")

## 11. Run on the dev contracts

`MODEL` is your OpenAI model name, for example `gpt-5-nano` or `gpt-4.1-nano`. First a free dry run with the fake model, then the real one.

In [ ]:
MODEL = "gpt-5-nano"
DEV = ["C01", "C02", "C11", "C12"]
cfg = RunConfig(model=MODEL, bounds=Bounds(
    confidence_threshold=0.75,
    max_steps_per_clause=4,
    max_tool_calls_per_contract=30,
    max_cost_usd_per_contract=0.05,
    max_tokens_per_contract=250_000,
))

fake_dir = run_agent(DEV, FakeLLM(), RunConfig(model="fake", bounds=cfg.bounds), out_dir="runs/fake_dev")

In [ ]:
llm = OpenAIChat(MODEL)
out_dir = run_agent(DEV, llm, cfg, out_dir=f"runs/dev_{MODEL}")

In [ ]:
import pandas as pd
res = pd.read_csv(out_dir / "results.csv")
display(pd.DataFrame(json.load(open(out_dir / "summary.json"))["contracts"]))
res[["contract_id", "family", "verdict", "severity", "confidence", "path", "steps", "evidence_verified", "section_ref"]]

## 12. Trace viewer

Click a family row to see each step. Forced stops are highlighted. Download `trace_viewer.html` to open it in a browser.

In [ ]:
from IPython.display import HTML, display
display(HTML((out_dir / "trace_viewer.html").read_text()))

## 13. Sanity check against the draft labels (dev only)

`Jenny_Labels_claude.xlsx` is an **AI draft**, not the gold standard. Use it only to spot obvious problems while tuning.

In [ ]:
draft = pd.read_excel(DATA_DIR / "Jenny_Labels_claude.xlsx", sheet_name="labels")[["contract_id", "family", "verdict", "severity"]]
cmp = res.merge(draft, on=["contract_id", "family"], suffixes=("_agent", "_draft"))
print("verdict agreement with draft:", round((cmp.verdict_agent == cmp.verdict_draft).mean(), 2))
cmp[["contract_id", "family", "verdict_agent", "verdict_draft", "severity_agent", "severity_draft", "path"]]

## 14. Questions queued for a human

In [ ]:
pd.DataFrame(json.load(open(out_dir / "human_queue.json")))

## 15. Download the run

In [ ]:
if IN_COLAB:
    !zip -qr run.zip {out_dir}
    from google.colab import files
    files.download("run.zip")